# Kandinsky 6 I2I — birchbark synth refinement

Interactive sandbox for refining synthetic carved birchbark images via the Kandinsky 6 I2I API. The matching dataset script lives at `scripts/refine_synth_with_kandinsky.py` and uses the same `KandinskyClient` defined in `src/birchbark_ocr/api/kandinsky.py`.

**Auth.** Set `KANDINSKY_TOKEN` in the env (e.g. `export KANDINSKY_TOKEN=...` before launching jupyter), or paste it into the `TOKEN` variable in cell 2. The token is sent as `Authorization: Bearer <token>`.

**Endpoints used** (kand_example.pdf):

* `POST /api/tasks/k6-i2i` — submit task, body `{params: {image: [base64], query: "..."}}`
* `GET  /api/tasks/<id>`        — poll status until `done`
* `GET  /api/tasks/<id>/result` — fetch the binary image

The PDF says generation takes ~10 s, so we poll every 3 s. We do **not** parallelize (single thread, polite cadence) — that's per the user requirement.

In [ ]:
from __future__ import annotations

import io
import json
import os
import sys
import time
from pathlib import Path

from PIL import Image

# Make the editable package importable even when this notebook is launched
# without `pip install -e .`.
ROOT = Path.cwd()
while ROOT != ROOT.parent and not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from birchbark_ocr.api import (
    KandinskyClient,
    KandinskyError,
    KandinskyTaskFailed,
    KandinskyTaskTimeout,
    match_input_aspect,
)

# TOKEN = os.environ.get("KANDINSKY_TOKEN", "").strip()
# if not TOKEN:
#     print("!! KANDINSKY_TOKEN is empty. Either export it or set TOKEN below.")
TOKEN = "HmTrdFKgmjKcMhBIfQcx"  # uncomment if not using the env var

BASE_URL = "https://studio.kandinskylab.ai/api"
DATASET_DIR = ROOT / "data/processed/synth_carved/dataset_a"
OUTPUT_DIR = ROOT / "reports/figs/kandinsky_refine_demo"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("ROOT       :", ROOT)
print("DATASET    :", DATASET_DIR)
print("OUTPUT     :", OUTPUT_DIR)
print("TOKEN set  :", bool(TOKEN))

In [ ]:
# Pick one carved sample from the dataset to use as a test input.
manifest_path = DATASET_DIR / "manifest.jsonl"
rows = [json.loads(l) for l in manifest_path.read_text().splitlines() if l.strip()]
ok = [r for r in rows if r.get("status") == "ok"]
print(f"manifest rows total: {len(rows)};  ok: {len(ok)}")

SAMPLE_INDEX = 33   # change to test a different image
sample = ok[SAMPLE_INDEX]
carved_path = Path(sample["carved_path"])
gold_path = Path(sample["gold_path"])
assert carved_path.exists(), carved_path

carved_img = Image.open(carved_path).convert("RGB")
gold_text = gold_path.read_text(encoding="utf-8") if gold_path.exists() else "(no gold)"
print("sample_id   :", sample.get("sample_id"))
print("carved size :", carved_img.size, "px")
print("gold (head) :", gold_text.splitlines()[:2])
carved_img

## Prompt

Kandinsky 6 I2I expects a free-form `query`. The job here is style transfer: the input is a clean rasterised carving on a partially weathered bark substrate; we want a museum-grade photograph of an old Novgorod birchbark gramota with the same letters, just more authentic.

Three default prompts to A/B. The first is concise (often best for I2I), the second adds period vocabulary, the third tries to guard against scene hallucination.

In [ ]:
PROMPT_TEXT = """\
Отредактируй это изображение берестяной грамоты так, чтобы оно выглядело как настоящая фотография археологической находки, пролежавшей в земле много столетий.
Главная цель правки — изменить ВИД БУКВ. Сейчас буквы выглядят как нарисованные тёмными чернилами поверх бересты. Нужно, чтобы они выглядели как старые, выцветшие порезы, прорезанные ножом или металлическим писалом в коре, то есть твоя задача - сделать буквы тонкими и выцветшими.

Изменения для БУКВ:
1. Каждая буква — это узкая БОРОЗДА, прорезанная В поверхности бересты, а не пигмент НА поверхности. Тёмный цвет штриха — это тень внутри прореза и осевшая в нём за столетия грязь, а не краска и не чернила. СДЕЛАЙ БУКВЫ ТОНКИМИ, КАК ВЫРЕЗАННЫМИ НОЖОМ, ПРЯМ ОЧЕНЬ ТОНКИМИ. Буквы в принипе НЕ ДОЛЖНЫ БЫТЬ РЕЛЬЕФНЫМИ, буквально толщина каждой буквы равно 0! Разрешаю тебе для этого менять форму букв.
2. Контраст букв с берестой должен быть СРЕДНИМ или СЛАБЫМ, а не сильным. Текст должен быть в целом читаем, но некоторые места — с трудом, как на настоящих древних грамотах.
3. Внутри одной буквы тон неравномерен: где штрих перерезает глубокое волокно — темнее, где идёт по поверхности — светлее. Не должно быть ощущения, что штрих «закрашен» равномерным тоном.
4. Края штрихов должны иметь естественную нерегулярность — маленькие зазубрины, лёгкие колебания толщины, места где лезвие зацепило волокно. При этом сами штрихи остаются ЧИТАЕМЫМИ, не размытыми и не смазанными.
5. НИКАКОГО эффекта объёма, выпуклости, тиснения, hightlight'а по краю буквы. Буквы уходят ВНУТРЬ поверхности, а не выпирают наружу.
6. Где буквы попадают на тёмные пятна, повреждения и потёртости бересты — они должны там быть частично потеряны, как будто повреждения возникли позже и съели часть текста. Это нужно сохранить и усилить.

Изменения для БЕРЕСТЫ:
7. Сохрани текущий вид бересты — пятна, патину, тёмные подтёки, текстуру волокон, шелушение. Этот аспект изображения уже удачный, его не надо переделывать, только аккуратно усилить.
8. Усиль ощущение, что береста — это древний органический материал, пролежавший в земле: лёгкие следы окисления, неравномерное потемнение по краям и вокруг повреждений.
9. Горизонтальные тёмные чёрточки и полосы (характерные для бересты чечевички и волокна) должны быть хорошо видны и проходить по всей поверхности.

ВАЖНЫЕ ОГРАНИЧЕНИЯ:
- НЕ убирай мелкие знаки — титла, точки, надстрочные буквы — 
  они должны остаться на своих местах.
- НЕ размывай и не растушёвывай штрихи — они выцветшие, но 
  всё ещё имеют различимые края.
- НЕ добавляй новых крупных повреждений, дыр или разрывов.
- НЕ меняй форму и размер самого куска бересты.
- НЕ меняй фон вокруг бересты.
"""


PROMPT = PROMPT_TEXT  # change between cells to A/B
print(PROMPT)

In [ ]:
client = KandinskyClient(
    token=TOKEN,
    base_url=BASE_URL,
    poll_interval=3.0,
    poll_timeout=180.0,
    timeout=60.0,
    max_retries=3,
)
print(client)

## Single end-to-end test

Submit one image, poll until `done`, fetch the result, save it next to the original. Each poll prints the status so you can see how long the queue actually takes.

In [ ]:
def on_poll(task_id, payload, elapsed):
    print(f"  [{elapsed:5.1f}s] task={task_id}  status={payload.get('status')}")

started = time.time()
task_id, final_status, body = client.i2i(
    image=carved_path,
    query=PROMPT,
    on_poll=on_poll,
)
elapsed = time.time() - started

# Raw Kandinsky output (whatever resolution it picks).
raw_img = Image.open(io.BytesIO(body)).convert("RGB")
raw_ext = client.guess_extension(body)
raw_path = OUTPUT_DIR / f"{sample['sample_id']}__kand_raw{raw_ext}"
raw_path.write_bytes(body)

# Aspect-correct it back to the input's proportions. Kandinsky 6 sometimes
# returns 1280x720 regardless of input shape, so wide synth inputs end up
# centered on a near-square canvas; match_input_aspect center-crops back.
fixed_img, aspect_info = match_input_aspect(
    raw_img,
    carved_img.size,
    tolerance=0.05,
    resize_to_input=True,
)
fixed_ext = ".png"
fixed_path = OUTPUT_DIR / f"{sample['sample_id']}__kand_fixed{fixed_ext}"
fixed_img.save(fixed_path)

print(f"task_id     : {task_id}")
print(f"elapsed     : {elapsed:.1f}s")
print(f"input size  : {carved_img.size}")
print(f"raw size    : {raw_img.size}  (aspect {aspect_info['output_aspect']})")
print(f"input aspect: {aspect_info['input_aspect']}")
print(f"aspect diff : {aspect_info['aspect_diff']:.4f}  "
      f"(tolerance {aspect_info['tolerance']:.2f})")
print(f"corrected   : {aspect_info['corrected']}")
print(f"cropped to  : {aspect_info['cropped_size']}")
print(f"final size  : {aspect_info['final_size']}")
print(f"raw saved   : {raw_path}")
print(f"fixed saved : {fixed_path}")
print(f"final task  : {json.dumps(final_status, ensure_ascii=False)[:300]}")
fixed_img

In [ ]:
# Show the input and the refined output side by side at matched height.
def side_by_side(images, labels, panel_h=320):
    thumbs = []
    for im in images:
        t = im.copy()
        t.thumbnail((4096, panel_h), Image.Resampling.LANCZOS)
        thumbs.append(t)
    pad = 12
    label_h = 22
    W = sum(t.width for t in thumbs) + pad * (len(thumbs) + 1)
    H = max(t.height for t in thumbs) + label_h + pad * 2
    out = Image.new("RGB", (W, H), (245, 245, 245))
    from PIL import ImageDraw
    draw = ImageDraw.Draw(out)
    x = pad
    for t, lbl in zip(thumbs, labels):
        out.paste(t, (x, label_h + pad))
        draw.text((x, pad // 2), lbl, fill=(40, 40, 40))
        x += t.width + pad
    return out

side_by_side(
    [carved_img, raw_img, fixed_img],
    [
        f"synth carved (input)  {carved_img.size}",
        f"kandinsky raw  {raw_img.size}",
        f"aspect-fixed  {fixed_img.size}"
        + ("  (corrected)" if aspect_info["corrected"] else "  (no fix needed)"),
    ],
)

### Notes / failure modes

* **Letters drift.** Kandinsky is a generative I2I, not pixel-faithful. Carved letters may be slightly redrawn. That is acceptable for OCR augmentation as long as the *content* of the text reads identical to the gold.
* **Wrong-aspect output.** Kandinsky 6 sometimes returns a fixed canvas (commonly 1280×720) regardless of input shape — most visible on very wide synth inputs (e.g. 1800×250). The notebook calls `match_input_aspect(...)` to center-crop the raw output back to the input aspect ratio and resample to the input dimensions. The dataset script does the same by default; pass `--no-aspect-fix` to disable, `--aspect-tolerance 0.10` to relax the trigger threshold, or `--save-raw` to also keep the un-corrected output for debugging.
* **Empty/non-image result body.** `KandinskyError("empty result body")` usually means the task is still on the API side; raise `poll_timeout` or rerun.
* **`KandinskyTaskTimeout`.** If timeouts happen often, raise `client.poll_timeout` (e.g. 300s) — the API doc says ~10s but queue depth varies.
* **HTTP 401/403.** Bad token. Re-export `KANDINSKY_TOKEN` and recreate the client.
* **HTTP 429.** Rate-limit. Lower the rate (you're already single-threaded; add an extra `time.sleep(2)` between submissions in the dataset script).
* **HTTP 5xx with retries exhausted.** Server side. Manual retry tomorrow.

Once a prompt is dialled in, run `scripts/refine_synth_with_kandinsky.py --dataset-name dataset_a --query "..."` from a screen session — it uses this same `KandinskyClient` and the same aspect-fix.